# PYTHIA standalone tune comparison

Compare the standalone PYTHIA CP5 and CUEP8M1 generator analyses from `$HOME/cernbox/ana/pPb8160/pythia/{cp5tune,cuep8m1tune}/`. The workflow overlays the two tunes and draws CUEP8M1/CP5 ratios for inclusive-jet and dijet full-$\eta_{CM}$ distributions and their Forward/Backward distributions. It also compares inclusive-jet $p_T$ spectra in five $\eta_{CM}$ windows, event weights, separate unweighted and weighted $\hat p_T$ distributions, and the dijet $p_T^{ave}/\hat p_T$ response versus $\hat p_T$.

### Physics and statistical conventions

All projections use half-open momentum intervals $[p_{T,low},p_{T,high})$. Full $\eta$ distributions are normalized independently according to `NORMALIZATION`; the default unit bin-content sum compares shapes. F/B is always constructed from the **unnormalized** forward and backward weighted yields folded from the full distribution. Both F/B construction and the later CUEP8M1/CP5 comparison use `TH1::Divide(..., '')` with independent-error propagation; binomial errors are neither used nor configurable. For these equal-energy pp collisions, $\eta_{CM}=\eta_{lab}$.

Input resolution uses reusable files for the requested jet-selection stem, then the original ROOT file, and raises an error if neither exists. Source eta-cut tables must match the current producer; stale inputs require regeneration. `ETA_CUT_VALUE` (where present) is the physical per-jet acceptance, default 1.9; its index is derived from the shared table. `SINGLE_JET_BINNING` and `DIJET_BINNING` independently select existing `normal` or `test` intervals. Existing `PTAVE_BIN_SET` controls use `standard` as the name for normal binning. Trigger-specific normal bins remain the notebook defaults; test dijet bins use MB 50–100, Jet80 100–180, and Jet100 above 180 GeV. Jet60 has no test interval. Diagnostic frame scans and trigger-stitching overlap windows retain their established definitions.

## Environment and imports

Start Jupyter from the repository root with `py-env/bin/python -m jupyter notebook`.

In [ ]:
# Cell role: initialize the project Python/PyROOT environment and shared helpers.
# Interpretation: no physics object is changed; ROOT ownership is configured before input is opened.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys
from IPython.display import display

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'CMakeLists.txt').is_file() and (p / 'hist_analysis').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Cannot locate jetAnalysis. Start Jupyter from its root.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)
from hist_analysis.config.histograms import DIJET_DELTA_PHI_SELECTION_LABEL, DIJET_PTAVE_BINS, STANDARD_DIJET_ETA_CUT_INDEX, TEST_DIJET_PTAVE_BINS
from hist_analysis.python.data_distributions import forward_backward_from_full
from hist_analysis.python.dijet_closures import DijetClosureCurve, build_dijet_gen_comparisons
from hist_analysis.python.event_plots import draw_event_histograms, draw_overweight_map
from hist_analysis.python.histogram_io import load_histogram
from hist_analysis.python.histogram_ops import normalize_histogram
from hist_analysis.python.mplhep_plotting import MplhepStyle, save_2d as save_mplhep_2d, save_comparison as save_mplhep_comparison
from hist_analysis.python.plot_labels import CMS_SIMULATION_HEADER, PP_HEADER_RIGHT
from hist_analysis.python.plotting import draw_closure, draw_overlay
from hist_analysis.python.projections import project_semantic_th2
from hist_analysis.python.histogram_production import resolve_catalog_file
from hist_analysis.python.root_style import PlotStyle, configure_cms_header

ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

## Configuration and input resolution

Both tunes are enabled by default with `TUNES`; select either one alone when only a single-sample check is wanted. `CP5_GEN_ANALYSIS_FILE` and `CUEP8M1_GEN_ANALYSIS_FILE` override the defaults. Index 5 is the reference per-jet acceptance $|\eta_{CM}^{jet}|<1.9$.

In [ ]:
SINGLE_JET_BINNING = 'normal'  # normal or test; independent of dijet binning
ETA_CUT_VALUE = 1.9  # per-jet |etaCM| <= value; supported producer cuts only
from hist_analysis.config.histograms import pt_bins, eta_cut_index
# Cell role: define user-facing inputs, binning, normalization, and plotting controls.
# Interpretation: these settings determine the selected standalone tunes and displayed phase-space intervals.
TUNES = ('CP5', 'CUEP8M1')  # choose either tune alone or both for comparisons
INCLUSIVE_PT_BINS = tuple(pt_bins('single', SINGLE_JET_BINNING))
INCLUSIVE_ETA_WINDOWS = ((-3.2, -2.4), (-2.4, -1.4), (-1.4, 1.4), (1.4, 2.4), (2.4, 3.2))
PTAVE_BIN_SET = 'test'      # test or standard
from hist_analysis.config.histograms import DIJET_ETA_CUTS, STANDARD_DIJET_ETA_CUT_INDEX
ETA_CUTS = DIJET_ETA_CUTS  # shared current production cut order
ETA_CUT_INDEX = eta_cut_index(ETA_CUT_VALUE)
REBIN_INCLUSIVE_ETA = 2
REBIN_DIJET_ETA = 2
NORMALIZATION = 'integral'  # none, integral, or bin_width; never applied before F/B
FULL_ETA_Y_RANGE = None  # optional common range for every inclusive and dijet full-eta plot
INCLUSIVE_Y_RANGE = None
INCLUSIVE_FB_Y_RANGE = (0.75, 1.40)
INCLUSIVE_RATIO_Y_RANGE = (0.75, 1.25)
INCLUSIVE_FB_RATIO_Y_RANGE = (0.75, 1.25)
INCLUSIVE_PT_SPECTRUM_RANGE = (30.0, 500.0)
INCLUSIVE_PT_RATIO_Y_RANGE = (0.75, 1.25)
EVENT_WEIGHT_X_RANGE = (-0.1, 1.1)
EVENT_WEIGHT_RATIO_Y_RANGE = (0.75, 1.25)
PTHAT_RATIO_Y_RANGE = (0.75, 1.25)
PTHAT_RESPONSE_X_RANGE = (0.0, 1200.0)
PTHAT_RESPONSE_MAP_Y_RANGE = (0.0, 20.0)
PTHAT_RESPONSE_MEAN_Y_RANGE = (0.0, 2.0)
FIT_SLICES_MIN_ENTRIES = 10
CONTROL_NORMALIZATION = 'integral'  # none, integral, or bin_width
DIJET_Y_RANGE = (0.00, 0.14)  # common range for every full dijet eta distribution
DIJET_OVERLAY_Y_RANGE = None
FB_Y_RANGE = (0.75, 1.25)
DIJET_RATIO_Y_RANGE = (0.75, 1.25)
DIJET_FB_RATIO_Y_RANGE = (0.75, 1.25)
DRAW_GRID = True
SAVE_PNG = False
SAVE_MPLHEP = True  # save parallel 8 x 8 inch CMS-style figures; ROOT outputs are always retained
SAVE_MPLHEP_PNG = False
MPLHEP_CMS_HEADER_LEFT = CMS_SIMULATION_HEADER  # CMS Simulation, CMS Preliminary, or CMS
MPLHEP_CMS_HEADER_RIGHT = PP_HEADER_RIGHT  # or r'pPb 174.6 nb$^{-1}$ (8.16 TeV)'
ROOT_PLOT_STYLE = PlotStyle(
    canvas_width=800, canvas_height=800, top_margin=0.12,
    annotation_text_size=0.040, legend_text_size=0.040,
    single_axis_title_size=0.052, single_axis_label_size=0.046,
    ratio_top_axis_title_size=0.056, ratio_top_axis_label_size=0.048,
    ratio_axis_title_size=0.120, ratio_axis_label_size=0.110,
    cms_header_text_size=0.040, cms_header_y=0.985,
)
ROOT_CURVE_STYLES = {'CP5': (ROOT.kRed + 1, 20), 'CUEP8M1': (ROOT.kBlue + 1, 24)}
ROOT_PLOT_KWARGS = {
    'style': ROOT_PLOT_STYLE, 'curve_styles': ROOT_CURVE_STYLES,
    'cms_header_left': MPLHEP_CMS_HEADER_LEFT, 'cms_header_right': MPLHEP_CMS_HEADER_RIGHT,
}
configure_cms_header(MPLHEP_CMS_HEADER_LEFT, MPLHEP_CMS_HEADER_RIGHT)
ROOT_COMPARISON_LAYOUT = {
    'annotation_x': 0.22, 'annotation_y': 0.88,
    'legend_bounds': (0.70, 0.72, 0.88, 0.85),
}
MPLHEP_STYLE = MplhepStyle(
    figure_size=8.0, axis_title_size=22.0, ratio_axis_title_size=17.0,
    tick_label_size=17.0, ratio_tick_label_size=15.0,
    annotation_text_size=16.0, annotation_y=0.955,
    legend_text_size=17.0, cms_label_size=20.0,
    marker_size=10.0, ratio_marker_size=9.0, capsize=2.0,
    colorbar_tick_size=14.0,
)
OUTPUT_DIR = Path(os.environ.get('PYTHIA_STANDALONE_CHECK_OUTPUT_DIR', PROJECT_ROOT / 'hist_analysis' / 'output' / 'pythia_standalone_check'))
MPLHEP_OUTPUT_DIR = OUTPUT_DIR / 'mplhep'

DEFAULT_FILES = {
    'CP5': resolve_catalog_file('standalone__cp5__pp', require_exists=False),
    'CUEP8M1': resolve_catalog_file('standalone__cuep8m1__pp', require_exists=False),
}
ENVIRONMENT_OVERRIDES = {
    'CP5': 'CP5_GEN_ANALYSIS_FILE',
    'CUEP8M1': 'CUEP8M1_GEN_ANALYSIS_FILE',
}

if not TUNES or len(set(TUNES)) != len(TUNES) or any(tune not in DEFAULT_FILES for tune in TUNES):
    raise ValueError(f'TUNES must contain unique entries selected from {tuple(DEFAULT_FILES)}')
if PTAVE_BIN_SET not in {'test', 'standard'}:
    raise ValueError("PTAVE_BIN_SET must be 'test' or 'standard'")
if NORMALIZATION not in {'none', 'integral', 'bin_width'}:
    raise ValueError("NORMALIZATION must be 'none', 'integral', or 'bin_width'")
if CONTROL_NORMALIZATION not in {'none', 'integral', 'bin_width'}:
    raise ValueError("CONTROL_NORMALIZATION must be 'none', 'integral', or 'bin_width'")
if FULL_ETA_Y_RANGE is not None and (len(FULL_ETA_Y_RANGE) != 2 or FULL_ETA_Y_RANGE[0] >= FULL_ETA_Y_RANGE[1]):
    raise ValueError('FULL_ETA_Y_RANGE must be None or an ordered (low, high) pair')
if MPLHEP_CMS_HEADER_LEFT not in {'CMS Simulation', 'CMS Preliminary', 'CMS'}:
    raise ValueError('MPLHEP_CMS_HEADER_LEFT must be CMS Simulation, CMS Preliminary, or CMS')
if not isinstance(MPLHEP_CMS_HEADER_RIGHT, str) or not MPLHEP_CMS_HEADER_RIGHT.strip():
    raise ValueError('MPLHEP_CMS_HEADER_RIGHT must be a non-empty string')
if ETA_CUT_INDEX not in range(len(ETA_CUTS)):
    raise ValueError(f'ETA_CUT_INDEX must be in [0, {len(ETA_CUTS)-1}]')
for label, intervals in {'INCLUSIVE_PT_BINS': INCLUSIVE_PT_BINS, 'INCLUSIVE_ETA_WINDOWS': INCLUSIVE_ETA_WINDOWS}.items():
    if not intervals or any(low >= high for low, high in intervals):
        raise ValueError(f'{label} must contain intervals satisfying low < high')
for label, value in {'REBIN_INCLUSIVE_ETA': REBIN_INCLUSIVE_ETA, 'REBIN_DIJET_ETA': REBIN_DIJET_ETA}.items():
    if isinstance(value, bool) or not isinstance(value, int) or value < 1:
        raise ValueError(f'{label} must be a positive integer')

TUNE_FILES = {
    tune: Path(os.environ.get(ENVIRONMENT_OVERRIDES[tune], DEFAULT_FILES[tune])).expanduser()
    for tune in TUNES
}
missing_files = {tune: path for tune, path in TUNE_FILES.items() if not path.is_file()}
if missing_files:
    raise FileNotFoundError('Missing standalone ROOT file(s): ' + ', '.join(f'{tune}: {path}' for tune, path in missing_files.items()))

PTAVE_BINS = tuple(TEST_DIJET_PTAVE_BINS if PTAVE_BIN_SET == 'test' else DIJET_PTAVE_BINS)
ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]
for tune, path in TUNE_FILES.items():
    print(f'{tune} input:', path)
print(f'Per-jet eta cut: |etaCM| < {ETA_CUT:g} (index {ETA_CUT_INDEX})')
print(f'pTave bins ({PTAVE_BIN_SET}):', PTAVE_BINS)
SINGLE_JET_PT_BINS = pt_bins('single', SINGLE_JET_BINNING)


## Inclusive generator jets

Project `hGenInclusiveJetPtEtaCM` onto $\eta_{CM}$ for every configured half-open jet-$p_T$ interval. Each full-distribution curve is normalized independently according to `NORMALIZATION`. The corresponding F/B curve is folded from the raw projection before normalization: positive $\eta_{CM}$ supplies Forward and reflected negative $\eta_{CM}$ supplies Backward. Their stored weighted uncertainties are divided with ROOT's independent-error option. The standalone input contains only jets above its generation-time storage threshold (normally 30 GeV), so intervals must not be interpreted below that threshold.

In [ ]:
# Cell role: project inclusive etaCM and construct F/B in every configured jet-pT range.
# Interpretation: shape normalization is display-only; F/B uses the unnormalized weighted projections.
inclusive_sources = {}
for tune, filename in TUNE_FILES.items():
    source = load_histogram(str(filename), 'hGenInclusiveJetPtEtaCM')
    if not source.InheritsFrom('TH2'):
        raise TypeError(f'hGenInclusiveJetPtEtaCM in {filename} must be a TH2')
    source.Rebin2D(1, REBIN_INCLUSIVE_ETA)
    inclusive_sources[tune] = source

inclusive_normalization_titles = {'none': 'Weighted jets', 'integral': '1/N dN/d#eta_{CM}', 'bin_width': '1/N dN/d#eta_{CM}'}
dijet_normalization_titles = {'none': 'Weighted dijets', 'integral': '1/N dN/d#eta_{CM}^{dijet}', 'bin_width': '1/N dN/d#eta_{CM}^{dijet}'}
inclusive_results = {}
for index, (low, high) in enumerate(INCLUSIVE_PT_BINS):
    raw = {
        tune: project_semantic_th2(source, 'eta', (low, high), name=f'{tune.lower()}_inclusive_eta_{index}')
        for tune, source in inclusive_sources.items()
    }
    shapes = {tune: normalize_histogram(histogram, NORMALIZATION) for tune, histogram in raw.items()}
    fb = {tune: forward_backward_from_full(histogram, name=f'{tune.lower()}_inclusive_fb_{index}') for tune, histogram in raw.items()}
    tag = f'{low:g}_{high:g}'.replace('.', 'p')
    annotations = (
        'PYTHIA 8.318', 'Inclusive jets',
        'Jet selection: anti-k_{T} R=0.4',
        f'{low:g} < p_{{T}}^{{jet}} < {high:g} GeV',
    )
    if len(TUNES) == 2:
        eta_canvas, eta_ratios = draw_closure(
            shapes, 'CP5', title='', x_title='#eta_{CM}', y_title=inclusive_normalization_titles[NORMALIZATION],
            ratio_range=INCLUSIVE_RATIO_Y_RANGE, ratio_y_title='CUEP8M1 / CP5', ratio_option='',
            draw_nominal_ratio=False, x_range=(-4.0, 4.0), y_range=FULL_ETA_Y_RANGE or INCLUSIVE_Y_RANGE, grid=DRAW_GRID,
            annotations=annotations, headroom=1.6,
            output=OUTPUT_DIR / f'pythia_gen_inclusive_eta_pt_{tag}.pdf',
            save_png=SAVE_PNG, canvas_name=f'pythia_gen_inclusive_eta_pt_{tag}',
            **ROOT_COMPARISON_LAYOUT, **ROOT_PLOT_KWARGS,
        )
        fb_canvas, fb_tune_ratios = draw_closure(
            fb, 'CP5', title='', x_title='|#eta_{CM}|', y_title='Forward / Backward',
            ratio_range=INCLUSIVE_FB_RATIO_Y_RANGE, ratio_y_title='CUEP8M1 / CP5', ratio_option='',
            draw_nominal_ratio=False, x_range=(0.0, 4.0), y_range=INCLUSIVE_FB_Y_RANGE, grid=DRAW_GRID,
            annotations=annotations,
            output=OUTPUT_DIR / f'pythia_gen_inclusive_fb_pt_{tag}.pdf',
            save_png=SAVE_PNG, canvas_name=f'pythia_gen_inclusive_fb_pt_{tag}',
            **ROOT_COMPARISON_LAYOUT, **ROOT_PLOT_KWARGS,
        )
    else:
        eta_canvas = draw_overlay(shapes, title='', x_title='#eta_{CM}', y_title=inclusive_normalization_titles[NORMALIZATION], x_range=(-4.0, 4.0), y_range=FULL_ETA_Y_RANGE or INCLUSIVE_Y_RANGE, grid=DRAW_GRID, annotations=annotations, output=OUTPUT_DIR / f'{TUNES[0].lower()}_gen_inclusive_eta_pt_{tag}.pdf', save_png=SAVE_PNG, canvas_name=f'{TUNES[0].lower()}_gen_inclusive_eta_pt_{tag}', **ROOT_PLOT_KWARGS)
        fb_canvas = draw_overlay(fb, title='', x_title='|#eta_{CM}|', y_title='Forward / Backward', x_range=(0.0, 4.0), y_range=INCLUSIVE_FB_Y_RANGE, grid=DRAW_GRID, reference_y=1.0, annotations=annotations, output=OUTPUT_DIR / f'{TUNES[0].lower()}_gen_inclusive_fb_pt_{tag}.pdf', save_png=SAVE_PNG, canvas_name=f'{TUNES[0].lower()}_gen_inclusive_fb_pt_{tag}', **ROOT_PLOT_KWARGS)
        eta_ratios, fb_tune_ratios = {}, {}
    inclusive_results[(low, high)] = {'eta': shapes, 'fb': fb, 'eta_ratios': eta_ratios, 'fb_tune_ratios': fb_tune_ratios}
    display(eta_canvas)
    display(fb_canvas)

## Inclusive-jet $p_T$ spectra in $\eta_{CM}$ windows

Project the weighted inclusive-jet histogram onto $p_T$ in the five configured half-open pseudorapidity windows. Each tune is normalized independently according to `CONTROL_NORMALIZATION`; the lower panel shows CUEP8M1/CP5 with independent-error propagation.

In [ ]:
# Cell role: compare inclusive-jet pT spectra between tunes in five etaCM windows.
# Interpretation: each displayed tune spectrum is normalized independently; ratios compare the resulting shapes.
def draw_tune_comparison(histograms, *, x_title, y_title, ratio_range, annotations, output, canvas_name, x_range=None, log_y=False):
    if len(TUNES) == 2:
        return draw_closure(
            histograms, 'CP5', title='', x_title=x_title, y_title=y_title,
            ratio_range=ratio_range, ratio_y_title='CUEP8M1 / CP5', ratio_option='',
            draw_nominal_ratio=False, x_range=x_range, log_y=log_y, grid=DRAW_GRID,
            annotations=annotations, headroom=1.8,
            output=output, save_png=SAVE_PNG, canvas_name=canvas_name,
            **ROOT_COMPARISON_LAYOUT, **ROOT_PLOT_KWARGS,
        )
    canvas = draw_overlay(
        histograms, title='', x_title=x_title, y_title=y_title, x_range=x_range,
        log_y=log_y, grid=DRAW_GRID, annotations=annotations,
        output=output, save_png=SAVE_PNG, canvas_name=canvas_name,
        **ROOT_PLOT_KWARGS,
    )
    return canvas, {}

control_y_titles = {
    'none': 'Weighted yield', 'integral': 'Normalized yield', 'bin_width': 'Normalized density',
}
inclusive_pt_results = {}
for index, (eta_low, eta_high) in enumerate(INCLUSIVE_ETA_WINDOWS):
    spectra = {
        tune: normalize_histogram(
            project_semantic_th2(source, 'pt', (eta_low, eta_high), name=f'{tune.lower()}_inclusive_pt_eta_{index}'),
            CONTROL_NORMALIZATION,
        )
        for tune, source in inclusive_sources.items()
    }
    tag = f'{eta_low:g}_{eta_high:g}'.replace('-', 'm').replace('.', 'p')
    annotations = (
        'PYTHIA 8.318', 'Inclusive jets',
        'Jet selection: anti-k_{T} R=0.4',
        f'{eta_low:g} #leq #eta_{{CM}} < {eta_high:g}',
    )
    canvas, ratios = draw_tune_comparison(
        spectra, x_title='p_{T}^{jet} (GeV)', y_title=control_y_titles[CONTROL_NORMALIZATION],
        ratio_range=INCLUSIVE_PT_RATIO_Y_RANGE, annotations=annotations,
        x_range=INCLUSIVE_PT_SPECTRUM_RANGE, log_y=True,
        output=OUTPUT_DIR / f'pythia_gen_inclusive_pt_eta_{tag}.pdf',
        canvas_name=f'pythia_gen_inclusive_pt_eta_{tag}',
    )
    inclusive_pt_results[(eta_low, eta_high)] = {'spectra': spectra, 'ratios': ratios, 'canvas': canvas}
    display(canvas)

## Event weights and $\hat p_T$

Compare the stored `hEventWeight` distributions and the generator $\hat p_T$ distributions. Weighted `hPTHat` and unweighted `hPTHatUnweighted` are shown on separate canvases. Every displayed distribution is normalized independently according to `CONTROL_NORMALIZATION`, and CUEP8M1/CP5 uses independent-error propagation.

In [ ]:
# Cell role: compare eventWeight and separate weighted/unweighted ptHat distributions.
# Interpretation: these normalized shape checks expose differences in generator sampling and event weighting.
control_specs = (
    ('event_weight', 'hEventWeight', 'Event weight', EVENT_WEIGHT_X_RANGE, EVENT_WEIGHT_RATIO_Y_RANGE, ('PYTHIA 8.318', 'Event weights')),
    ('pthat_unweighted', 'hPTHatUnweighted', '#hat{p}_{T} (GeV)', None, PTHAT_RATIO_Y_RANGE, ('PYTHIA 8.318', 'Unweighted #hat{p}_{T}')),
    ('pthat_weighted', 'hPTHat', '#hat{p}_{T} (GeV)', None, PTHAT_RATIO_Y_RANGE, ('PYTHIA 8.318', 'Weighted #hat{p}_{T}')),
)
event_control_results = {}
for tag, key, x_title, x_range, ratio_range, annotations in control_specs:
    histograms = {
        tune: normalize_histogram(load_histogram(str(filename), key), CONTROL_NORMALIZATION)
        for tune, filename in TUNE_FILES.items()
    }
    canvas, ratios = draw_tune_comparison(
        histograms, x_title=x_title, y_title=control_y_titles[CONTROL_NORMALIZATION],
        ratio_range=ratio_range, annotations=annotations, x_range=x_range, log_y=True,
        output=OUTPUT_DIR / f'pythia_{tag}.pdf', canvas_name=f'pythia_{tag}',
    )
    event_control_results[tag] = {'histograms': histograms, 'ratios': ratios, 'canvas': canvas}
    display(canvas)

## Dijet response versus $\hat p_T$

Draw the weighted `hGenDijetPtAveOverPTHatVsPTHat` map for each enabled tune. The map contains the two leading stored jets before the final 50/40 GeV and $|\Delta\phi|$ dijet cuts, so large $p_T^{ave}/\hat p_T$ values expose rare upward migrations. `TH2::FitSlicesY` fits a Gaussian to each $\hat p_T$ slice and extracts parameter 1, the fitted response mean. A slice must satisfy `FIT_SLICES_MIN_ENTRIES`; failed or insufficient fits remain empty. The fit means use the stored weighted histogram and therefore diagnose the response of the weighted sample rather than an unweighted event population.


In [ ]:
# Cell role: draw weighted pTave/pTHat versus pTHat maps and extract Gaussian slice means.
# Interpretation: large ratios reveal upward migrations; fitted means summarize each weighted pTHat slice.
def fit_slice_means(histogram, *, name):
    if not histogram.InheritsFrom('TH2'):
        raise TypeError(f'Expected TH2, got {histogram.ClassName()}')
    fit_results = ROOT.TObjArray()
    histogram.FitSlicesY(
        ROOT.nullptr, 1, histogram.GetNbinsX(), FIT_SLICES_MIN_ENTRIES, 'QNR', fit_results,
    )
    fitted_mean = fit_results.At(1)
    if not fitted_mean:
        raise RuntimeError(f'FitSlicesY did not produce a mean histogram for {histogram.GetName()}')
    fitted_mean = fitted_mean.Clone(name)
    fitted_mean.SetDirectory(0)
    fitted_mean.SetTitle('')
    fitted_mean.GetXaxis().SetRangeUser(*PTHAT_RESPONSE_X_RANGE)
    return fitted_mean

pthat_response_maps = {}
pthat_response_means = {}
for tune, filename in TUNE_FILES.items():
    response = load_histogram(str(filename), 'hGenDijetPtAveOverPTHatVsPTHat')
    if not response.InheritsFrom('TH2'):
        raise TypeError(f'hGenDijetPtAveOverPTHatVsPTHat in {filename} must be a TH2')
    response.GetXaxis().SetRangeUser(*PTHAT_RESPONSE_X_RANGE)
    response.GetYaxis().SetRangeUser(*PTHAT_RESPONSE_MAP_Y_RANGE)
    canvas = draw_overweight_map(
        response, title=f'{tune}: Gen dijet response',
        generator_label='PYTHIA 8 standalone', orientation_label=f'{tune}, pp 8.16 TeV',
        grid=DRAW_GRID,
        output=OUTPUT_DIR / f'{tune.lower()}_gen_dijet_ptave_over_pthat_vs_pthat.pdf',
        save_png=SAVE_PNG,
    )
    pthat_response_maps[tune] = {'histogram': response, 'canvas': canvas}
    pthat_response_means[tune] = fit_slice_means(
        response, name=f'h{tune}GenDijetPtAveOverPTHatMean',
    )
    display(canvas)

pthat_response_mean_canvas, pthat_response_mean_plots = draw_event_histograms(
    pthat_response_means, title='Gen dijet response mean',
    x_title='#hat{p}_{T} (GeV)', y_title='Gaussian mean of p_{T}^{ave}/#hat{p}_{T}',
    y_range=PTHAT_RESPONSE_MEAN_Y_RANGE, normalization='none', grid=DRAW_GRID,
    generator_label='PYTHIA 8 standalone', orientation_label='pp 8.16 TeV',
    output=OUTPUT_DIR / 'pythia_gen_dijet_ptave_over_pthat_fit_slices_mean.pdf',
    save_png=SAVE_PNG,
)
display(pthat_response_mean_canvas)


## Generator dijets and Forward/Backward ratios

For each selected $p_T^{ave}$ interval and tune, the shared closure helper projects `hGenDijetPtEtaCM_{index}` and its stored `Forward` and `Backward` companions over identical bins. Full distributions are normalized independently for display. Forward and backward remain unnormalized and are divided with ROOT's independent-error option `''`; CUEP8M1/CP5 ratios are then drawn for both observables with the same independent-error convention.

In [ ]:
# Cell role: build the full dijet eta shape and unnormalized-yield F/B ratio for every pTave interval.
# Interpretation: F/B tests pp forward/backward symmetry; it is not an efficiency and never uses binomial errors.
GEN_CURVE = DijetClosureCurve(
    'Gen', 'hGenDijetPtEtaCM_{eta_cut_index}',
)
DIJET_CACHE = {}
for ptave_range in PTAVE_BINS:
    entry = {'eta': {}, 'fb': {}, 'keys': {}}
    for tune, filename in TUNE_FILES.items():
        eta_shapes, fb_ratios, selected_keys = build_dijet_gen_comparisons(
            filename, (GEN_CURVE,), eta_cut_index=ETA_CUT_INDEX, ptave_range=ptave_range,
            nominal='Gen', rebin_eta=REBIN_DIJET_ETA, normalization=NORMALIZATION, ratio_option='',
        )
        entry['eta'][tune] = eta_shapes['Gen']
        entry['fb'][tune] = fb_ratios['Gen']
        entry['keys'][tune] = selected_keys['Gen']
    DIJET_CACHE[tuple(ptave_range)] = entry
reference_dijet = next(iter(next(iter(DIJET_CACHE.values()))['eta'].values()))
reference_dijet_axis = reference_dijet.GetXaxis()
cut_bin = reference_dijet_axis.FindBin(ETA_CUT - 1e-6)
DIJET_DISPLAY_EDGE = ETA_CUT + reference_dijet_axis.GetBinWidth(cut_bin)
DIJET_FULL_X_RANGE = (-DIJET_DISPLAY_EDGE, DIJET_DISPLAY_EDGE)
DIJET_FB_X_RANGE = (0.0, DIJET_DISPLAY_EDGE)
print(f'Prepared {len(DIJET_CACHE)} dijet intervals for {TUNES}')
print(f'Dijet display range includes the final selected bin through |eta| = {DIJET_DISPLAY_EDGE:g}')

In [ ]:
# Cell role: overlay standalone tunes and draw CUEP8M1/CP5 ratios for full etaCM and F/B.
# Interpretation: F/B and tune-to-tune ratios both use independent weighted-sample uncertainties.
dijet_results = {}
for low, high in PTAVE_BINS:
    entry = DIJET_CACHE[(low, high)]
    tag = f'{low:g}_{high:g}'.replace('.', 'p')
    annotations = (
        'PYTHIA 8.318', 'Gen dijets',
        f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV',
        f'|#eta_{{CM}}^{{jet}}| < {ETA_CUT:g}',
        'p_{T}^{Lead} #geq 50 GeV', 'p_{T}^{SubLead} #geq 40 GeV',
        DIJET_DELTA_PHI_SELECTION_LABEL,
    )
    if len(TUNES) == 2:
        eta_canvas, eta_ratios = draw_closure(
            entry['eta'], 'CP5', title='', x_title='#eta_{CM}^{dijet}',
            y_title=dijet_normalization_titles[NORMALIZATION], ratio_range=DIJET_RATIO_Y_RANGE,
            ratio_y_title='CUEP8M1 / CP5', ratio_option='', draw_nominal_ratio=False,
            x_range=DIJET_FULL_X_RANGE, y_range=FULL_ETA_Y_RANGE or DIJET_Y_RANGE, grid=DRAW_GRID,
            annotations=annotations, headroom=1.6,
            output=OUTPUT_DIR / f'pythia_gen_dijet_eta_ptave_{tag}.pdf', save_png=SAVE_PNG,
            canvas_name=f'pythia_gen_dijet_eta_ptave_{tag}',
            **ROOT_COMPARISON_LAYOUT, **ROOT_PLOT_KWARGS,
        )
        fb_canvas, fb_tune_ratios = draw_closure(
            entry['fb'], 'CP5', title='', x_title='|#eta_{CM}^{dijet}|', y_title='Forward / Backward',
            ratio_range=DIJET_FB_RATIO_Y_RANGE, ratio_y_title='CUEP8M1 / CP5', ratio_option='',
            draw_nominal_ratio=False, x_range=DIJET_FB_X_RANGE, y_range=FB_Y_RANGE, grid=DRAW_GRID,
            annotations=annotations,
            output=OUTPUT_DIR / f'pythia_gen_dijet_fb_ptave_{tag}.pdf',
            save_png=SAVE_PNG, canvas_name=f'pythia_gen_dijet_fb_ptave_{tag}',
            **ROOT_COMPARISON_LAYOUT, **ROOT_PLOT_KWARGS,
        )
    else:
        eta_canvas = draw_overlay(entry['eta'], title='', x_title='#eta_{CM}^{dijet}', y_title=dijet_normalization_titles[NORMALIZATION], x_range=DIJET_FULL_X_RANGE, y_range=FULL_ETA_Y_RANGE or DIJET_Y_RANGE, grid=DRAW_GRID, annotations=annotations, output=OUTPUT_DIR / f'{TUNES[0].lower()}_gen_dijet_eta_ptave_{tag}.pdf', save_png=SAVE_PNG, canvas_name=f'{TUNES[0].lower()}_gen_dijet_eta_ptave_{tag}', **ROOT_PLOT_KWARGS)
        fb_canvas = draw_overlay(entry['fb'], title='', x_title='|#eta_{CM}^{dijet}|', y_title='Forward / Backward', x_range=DIJET_FB_X_RANGE, y_range=FB_Y_RANGE, grid=DRAW_GRID, reference_y=1.0, annotations=annotations, output=OUTPUT_DIR / f'{TUNES[0].lower()}_gen_dijet_fb_ptave_{tag}.pdf', save_png=SAVE_PNG, canvas_name=f'{TUNES[0].lower()}_gen_dijet_fb_ptave_{tag}', **ROOT_PLOT_KWARGS)
        eta_ratios, fb_tune_ratios = {}, {}
    dijet_results[(low, high)] = {'eta_canvas': eta_canvas, 'fb_canvas': fb_canvas, 'eta_ratios': eta_ratios, 'fb_tune_ratios': fb_tune_ratios, **entry}
    display(eta_canvas)
    display(fb_canvas)

## Optional mplhep export

Set `SAVE_MPLHEP = True` to save parallel 8 x 8 inch figures using `mplhep`'s CMS style beneath `OUTPUT_DIR / 'mplhep'`. `MPLHEP_CMS_HEADER_LEFT` selects `CMS Simulation`, `CMS Preliminary`, or `CMS`; `MPLHEP_CMS_HEADER_RIGHT` sets the right label, for example `pp (8.16 TeV)` or `pPb 174.6 nb$^{-1}$ (8.16 TeV)`. This final export consumes the already-built detached ROOT histograms; it does not replace, alter, or suppress any ROOT canvas or ROOT-produced PDF. Set `SAVE_MPLHEP_PNG = True` to add 200 DPI PNG copies alongside the mplhep PDFs.

In [ ]:
# Cell role: optionally render all final histogram products with mplhep CMS styling.
# Interpretation: this is a display-only export of existing ROOT objects; no contents or errors are recalculated.
if SAVE_MPLHEP:
    def export_mplhep_comparison(*args, **kwargs):
        return save_mplhep_comparison(
            *args, style=MPLHEP_STYLE, cms_header_left=MPLHEP_CMS_HEADER_LEFT,
            cms_header_right=MPLHEP_CMS_HEADER_RIGHT, **kwargs,
        )

    def export_mplhep_2d(*args, **kwargs):
        return save_mplhep_2d(
            *args, style=MPLHEP_STYLE, cms_header_left=MPLHEP_CMS_HEADER_LEFT,
            cms_header_right=MPLHEP_CMS_HEADER_RIGHT, **kwargs,
        )

    mplhep_inclusive_y_titles = {
        'none': 'Weighted jets',
        'integral': r'$1/N\,dN/d\eta_{\mathrm{CM}}$',
        'bin_width': r'$1/N\,dN/d\eta_{\mathrm{CM}}$',
    }
    mplhep_dijet_y_titles = {
        'none': 'Weighted dijets',
        'integral': r'$1/N\,dN/d\eta_{\mathrm{CM}}^{\mathrm{dijet}}$',
        'bin_width': r'$1/N\,dN/d\eta_{\mathrm{CM}}^{\mathrm{dijet}}$',
    }
    mplhep_control_y_titles = {
        'none': 'Weighted yield', 'integral': 'Normalized yield', 'bin_width': 'Normalized density',
    }
    for (low, high), result in inclusive_results.items():
        tag = f'{low:g}_{high:g}'.replace('.', 'p')
        notes = ('PYTHIA 8.318', 'Inclusive jets', r'anti-$k_{\mathrm{T}}$ R=0.4', rf'${low:g} < p_{{\mathrm{{T}}}}^{{\mathrm{{jet}}}} < {high:g}$ GeV')
        export_mplhep_comparison(
            result['eta'], ratios=result['eta_ratios'], output=MPLHEP_OUTPUT_DIR / f'pythia_gen_inclusive_eta_pt_{tag}.pdf',
            x_label=r'$\eta_{\mathrm{CM}}$', y_label=mplhep_inclusive_y_titles[NORMALIZATION], x_range=(-4.0, 4.0),
            y_range=FULL_ETA_Y_RANGE or INCLUSIVE_Y_RANGE, ratio_range=INCLUSIVE_RATIO_Y_RANGE, annotations=notes, save_png=SAVE_MPLHEP_PNG,
        )
        export_mplhep_comparison(
            result['fb'], ratios=result['fb_tune_ratios'], output=MPLHEP_OUTPUT_DIR / f'pythia_gen_inclusive_fb_pt_{tag}.pdf',
            x_label=r'$|\eta_{\mathrm{CM}}|$', y_label='Forward / Backward', x_range=(0.0, 4.0), y_range=INCLUSIVE_FB_Y_RANGE,
            ratio_range=INCLUSIVE_FB_RATIO_Y_RANGE, annotations=notes, save_png=SAVE_MPLHEP_PNG,
        )

    for (eta_low, eta_high), result in inclusive_pt_results.items():
        tag = f'{eta_low:g}_{eta_high:g}'.replace('-', 'm').replace('.', 'p')
        export_mplhep_comparison(
            result['spectra'], ratios=result['ratios'], output=MPLHEP_OUTPUT_DIR / f'pythia_gen_inclusive_pt_eta_{tag}.pdf',
            x_label=r'$p_{\mathrm{T}}^{\mathrm{jet}}$ (GeV)', y_label=mplhep_control_y_titles[CONTROL_NORMALIZATION], x_range=INCLUSIVE_PT_SPECTRUM_RANGE,
            ratio_range=INCLUSIVE_PT_RATIO_Y_RANGE, log_y=True,
            annotations=('PYTHIA 8.318', 'Inclusive jets', r'anti-$k_{\mathrm{T}}$ R=0.4', rf'${eta_low:g} \leq \eta_{{\mathrm{{CM}}}} < {eta_high:g}$'),
            save_png=SAVE_MPLHEP_PNG,
        )

    control_ranges = {'event_weight': EVENT_WEIGHT_X_RANGE, 'pthat_unweighted': None, 'pthat_weighted': None}
    control_labels = {'event_weight': 'Event weight', 'pthat_unweighted': r'Unweighted $\hat{p}_{\mathrm{T}}$', 'pthat_weighted': r'Weighted $\hat{p}_{\mathrm{T}}$'}
    control_x_labels = {'event_weight': 'Event weight', 'pthat_unweighted': r'$\hat{p}_{\mathrm{T}}$ (GeV)', 'pthat_weighted': r'$\hat{p}_{\mathrm{T}}$ (GeV)'}
    for tag, result in event_control_results.items():
        export_mplhep_comparison(
            result['histograms'], ratios=result['ratios'], output=MPLHEP_OUTPUT_DIR / f'pythia_{tag}.pdf',
            x_label=control_x_labels[tag], y_label=mplhep_control_y_titles[CONTROL_NORMALIZATION], x_range=control_ranges[tag],
            ratio_range=EVENT_WEIGHT_RATIO_Y_RANGE if tag == 'event_weight' else PTHAT_RATIO_Y_RANGE, log_y=True,
            annotations=('PYTHIA 8.318', control_labels[tag]), save_png=SAVE_MPLHEP_PNG,
        )

    for tune, result in pthat_response_maps.items():
        export_mplhep_2d(
            result['histogram'], output=MPLHEP_OUTPUT_DIR / f'{tune.lower()}_gen_dijet_ptave_over_pthat_vs_pthat.pdf',
            x_label=r'$\hat{p}_{\mathrm{T}}$ (GeV)', y_label=r'$p_{\mathrm{T}}^{\mathrm{ave}} / \hat{p}_{\mathrm{T}}$', x_range=PTHAT_RESPONSE_X_RANGE,
            y_range=PTHAT_RESPONSE_MAP_Y_RANGE, annotation=f'PYTHIA 8.318, {tune}', save_png=SAVE_MPLHEP_PNG,
        )
    export_mplhep_comparison(
        pthat_response_mean_plots, output=MPLHEP_OUTPUT_DIR / 'pythia_gen_dijet_ptave_over_pthat_fit_slices_mean.pdf',
        x_label=r'$\hat{p}_{\mathrm{T}}$ (GeV)', y_label=r'Gaussian mean of $p_{\mathrm{T}}^{\mathrm{ave}} / \hat{p}_{\mathrm{T}}$',
        x_range=PTHAT_RESPONSE_X_RANGE, y_range=PTHAT_RESPONSE_MEAN_Y_RANGE, annotations=('PYTHIA 8.318', 'Gen dijets'),
        save_png=SAVE_MPLHEP_PNG,
    )

    for (low, high), result in dijet_results.items():
        tag = f'{low:g}_{high:g}'.replace('.', 'p')
        notes = (
            'PYTHIA 8.318', 'Gen dijets',
            rf'${low:g} < p_{{\mathrm{{T}}}}^{{\mathrm{{ave}}}} < {high:g}$ GeV',
            rf'$|\eta_{{\mathrm{{CM}}}}^{{\mathrm{{jet}}}}| < {ETA_CUT:g}$',
            r'$p_{\mathrm{T}}^{\mathrm{Lead}} \geq 50$ GeV',
            r'$p_{\mathrm{T}}^{\mathrm{SubLead}} \geq 40$ GeV',
            r'$|\Delta\phi| > 2\pi/3$',
        )
        export_mplhep_comparison(
            result['eta'], ratios=result['eta_ratios'], output=MPLHEP_OUTPUT_DIR / f'pythia_gen_dijet_eta_ptave_{tag}.pdf',
            x_label=r'$\eta_{\mathrm{CM}}^{\mathrm{dijet}}$', y_label=mplhep_dijet_y_titles[NORMALIZATION], x_range=DIJET_FULL_X_RANGE,
            y_range=FULL_ETA_Y_RANGE or DIJET_Y_RANGE, ratio_range=DIJET_RATIO_Y_RANGE, annotations=notes, save_png=SAVE_MPLHEP_PNG,
        )
        export_mplhep_comparison(
            result['fb'], ratios=result['fb_tune_ratios'], output=MPLHEP_OUTPUT_DIR / f'pythia_gen_dijet_fb_ptave_{tag}.pdf',
            x_label=r'$|\eta_{\mathrm{CM}}^{\mathrm{dijet}}|$', y_label='Forward / Backward', x_range=DIJET_FB_X_RANGE, y_range=FB_Y_RANGE,
            ratio_range=DIJET_FB_RATIO_Y_RANGE, annotations=notes, save_png=SAVE_MPLHEP_PNG,
        )
    print('Saved mplhep figures to', MPLHEP_OUTPUT_DIR)
else:
    print('mplhep export disabled; set SAVE_MPLHEP = True to enable it')